In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline




2026-06-11 20:13:52.668725: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-11 20:13:52.748560: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-11 20:13:54.805826: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# Langkah 2: Import Dataset

In [2]:
dataset = pd.read_csv('weatherAUS.csv')
pd.unique(dataset['RainTomorrow'])

array(['No', 'Yes', nan], dtype=object)

In [3]:
print("Distribusi Kelas Asli :\n", dataset['RainTomorrow'].value_counts())


Distribusi Kelas Asli :
 RainTomorrow
No     110316
Yes     31877
Name: count, dtype: int64


# Data Quality Audit


In [4]:
missing_report = pd.DataFrame({
    "missing_count" : dataset.isnull().sum(),
    "missing_percent": dataset.isnull().mean() * 100
}).sort_values(by="missing_percent", ascending=False)

missing_report

,missing_count,missing_percent
Sunshine,69835,48.009762
Evaporation,62790,43.166506
Cloud3pm,59358,40.807095
Cloud9am,55888,38.421559
Pressure9am,15065,10.356799
Pressure3pm,15028,10.331363
WindDir9am,10566,7.263853
WindGustDir,10326,7.098859
WindGustSpeed,10263,7.055548
Humidity3pm,4507,3.098446


In [5]:
dataset = dataset.dropna(subset=['RainTomorrow'])
dataset['Date'] = pd.to_datetime(dataset['Date'], errors='coerce')
dataset = dataset.dropna(subset=['Date'])

# reset dataset
dataset = dataset.reset_index(drop=True)

# Langkah 3: Pisahkan Fitur dan Target

In [6]:
x = dataset.drop(columns=['RainTomorrow', 'Date'])
y = dataset['RainTomorrow']



In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
x_train,x_test,y_train,y_test = train_test_split(x, y, test_size=0.2, random_state=42, shuffle=False)


x_train.shape, x_test.shape


((113754, 21), (28439, 21))

# Identifikasi kolom numerik dan kategorial

In [8]:
numeric_cols = x_train.select_dtypes(include=['number']).columns
categorial_cols = x_train.select_dtypes(include=['object']).columns


# Langkah 4: Melakukan Imputer & One Hot encode

# Imputer

In [9]:


numeric_imputer = IterativeImputer(random_state=42, initial_strategy='median', max_iter=20)
categorial_imputer = SimpleImputer(strategy='most_frequent')

In [10]:
x_train[numeric_cols] = numeric_imputer.fit_transform(x_train[numeric_cols])
x_train[categorial_cols] = categorial_imputer.fit_transform(x_train[categorial_cols])

x_test[numeric_cols] = numeric_imputer.transform(x_test[numeric_cols])
x_test[categorial_cols] = categorial_imputer.transform(x_test[categorial_cols])

/home/muhammad/MachineLearning/venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


# Encode Categorial 


In [11]:
# encoding fitur categorial

column_transfer = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
train_encode = column_transfer.fit_transform(x_train[categorial_cols])
cat_feature = column_transfer.get_feature_names_out(categorial_cols)

train_encode_df = pd.DataFrame(train_encode, columns=cat_feature, index=x_train.index)

# transform pada data test
test_encode = column_transfer.transform(x_test[categorial_cols])
test_encode_df = pd.DataFrame(test_encode, columns=cat_feature, index=x_test.index)

# gabung kolom baru dan hapus kategori lama
x_train = pd.concat([x_train.drop(columns=categorial_cols), train_encode_df], axis=1)
x_test = pd.concat([x_test.drop(columns=categorial_cols), test_encode_df], axis=1)

/home/muhammad/MachineLearning/venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


# Label Encode

In [12]:
label_encode = LabelEncoder()

y_train_encode = label_encode.fit_transform(y_train)
y_test_encode = label_encode.transform(y_test)



# Langkah 5: Feature Scalling

In [13]:
scale = StandardScaler()
x_train[numeric_cols] = scale.fit_transform(x_train[numeric_cols])
x_test[numeric_cols] = scale.fit_transform(x_test[numeric_cols])

# Langkah 6: Split atau Stratified Split

# Menggunakan Algoritma Random Forest Classifier

In [ ]:
model_rf = RandomForestClassifier( random_state=0)
model_rf.fit(x_train, y_train_encode)

# prediksi 
y_pred = model_rf.predict(x_test)

akurasi = accuracy_score(y_test_encode, y_pred)

print(f"Model akurasi {akurasi:4f}")


In [ ]:
model_rf = RandomForestClassifier(n_estimators=100, random_state=0, class_weight='balanced')
model_rf.fit(x_train, y_train_encode)

# prediksi 
y_pred = model_rf.predict(x_test)

akurasi = accuracy_score(y_test_encode, y_pred)

print(f"Model akurasi {akurasi:4f}")